# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

#### Finding 1 — Content Archetypes (k-means, k=5)

The paper clusters content into 5 archetypes. Three of the five clusters — Cluster 2, 3, and 4 — are all labeled "Rising Stars", despite having different health scores (48 / 47 / 43.2), impressions (928 / 3.9K / 2.0K), and word counts (2.3K / 689 / 2.3K).

- Label provenance: The paper calls this an "original heuristic label in the analysis script." If a rule-based labeling step assigns the same name to three structurally different clusters, that usually means the rule's thresholds aren't specific enough to separate them & not necessarily that the three clusters share a real business meaning.

- Validation design: The paper's own PCA plot caption notes cluster boundaries look "fuzzy rather than cleanly separated." Given that, I'd ask whether a silhouette score or other quantitative check backs k=5, or whether 5 was chosen mainly for interpretability.

My own w05 archetype-naming logic hits the identical failure — all 4 of my clusters resolve to the same label, HIGH_VOLUME_MIXED, for the same underlying reason. I'm flagging this because I found it in my own work first.

#### Finding 2 — What Predicts Health? (Random Forest feature importance)

The model ranks Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of Health Score.

- Label provenance: Health Score is defined earlier in the paper as a direct sum: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll (20 pts). Three of the model's top features are literally ingredients of the label it's predicting. The paper is upfront that "importance is descriptive rather than causal", but I'd ask: with ~90% of importance sitting on label-constituent features, is there evidence the model learned anything the scoring formula didn't already guarantee?

- Validation design: An 80/20 holdout split is reported, which protects against the model memorizing training rows. It does not protect against the label and the features sharing raw inputs, that risk survives any train/test split. I'd ask whether a version of the model was tested using only the non-constituent features (excluding position, impressions, CTR, and scroll), to see if it still beats chance.

This is the same shared-input risk I found between my own baseline score and clustering features (score correlates 0.70 with impressions, which is also a clustering input).

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My w05 model already used a grouped-by-client_hash_id split to choose k. To make the before/after comparison explicit for this audit, I re-ran the same K-Means model (k=4) on the same 182,135-row dataset under two splits:

- Before — naive random row split: the default most people reach for — 80/20 split with no regard for which client a row belongs to.

- After — grouped split by client (same as w05): an entire client's content lands in either train or test, never both.

Silhouette score is computed on a fixed 20,000-row subsample in both cases, so the comparison is fair and doesn't blow up runtime on 182K rows.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/baseline_action_score.csv')

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Clients: {df['client_hash_id'].nunique()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 182,135
Columns: 15
Clients: 59


In [5]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('/content/drive/MyDrive/baseline_action_score.csv')
df['impressions_log'] = np.log10(df['impressions'] + 1)
features = ['ctr', 'avg_position', 'impressions_log']
for col in features:
    df[col] = df[col].fillna(df[col].median())

X_full = df[features].values
k = 4
SIL_SAMPLE = 20000

def run_split(mask_train, mask_test, label):
    X_train, X_test = X_full[mask_train], X_full[mask_test]
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    train_labels = km.fit_predict(X_train_s)
    train_sil = silhouette_score(X_train_s, train_labels, sample_size=min(SIL_SAMPLE, len(X_train_s)), random_state=42)
    test_labels = km.predict(X_test_s)
    test_sil = silhouette_score(X_test_s, test_labels, sample_size=min(SIL_SAMPLE, len(X_test_s)), random_state=42)
    print(f"[{label}] train n={mask_train.sum():,}  test n={mask_test.sum():,}")
    print(f"[{label}] train_sil={train_sil:.3f}  test_sil={test_sil:.3f}  gap={train_sil-test_sil:+.3f}")
    return train_sil, test_sil

print("BEFORE -- naive row-level random split (ignores client grouping)")
np.random.seed(42)
n = len(df)
shuffled_idx = np.random.permutation(n)
split = int(0.8 * n)
mask_train_naive = np.zeros(n, dtype=bool); mask_train_naive[shuffled_idx[:split]] = True
mask_test_naive  = np.zeros(n, dtype=bool); mask_test_naive[shuffled_idx[split:]] = True
naive_train_sil, naive_test_sil = run_split(mask_train_naive, mask_test_naive, "NAIVE random split")

print("\nAFTER -- honest split, grouped by client_hash_id (same as w05)")
unique_clients = df['client_hash_id'].unique()
rng = np.random.RandomState(42)
shuffled_clients = rng.permutation(unique_clients)
split_idx = int(0.8 * len(shuffled_clients))
train_clients = set(shuffled_clients[:split_idx])
test_clients = set(shuffled_clients[split_idx:])
mask_train_grp = df['client_hash_id'].isin(train_clients).values
mask_test_grp  = df['client_hash_id'].isin(test_clients).values
grp_train_sil, grp_test_sil = run_split(mask_train_grp, mask_test_grp, "GROUPED (honest) split")

comp = pd.DataFrame({
    'split_type': ['Naive random (before)', 'Grouped by client (after)'],
    'train_sil': [naive_train_sil, grp_train_sil],
    'test_sil': [naive_test_sil, grp_test_sil],
    'train_minus_test_gap': [naive_train_sil - naive_test_sil, grp_train_sil - grp_test_sil],
})
print()
print(comp.to_string(index=False))

train_clients_seen = set(df.loc[mask_train_naive, 'client_hash_id'].unique())
test_clients_seen  = set(df.loc[mask_test_naive, 'client_hash_id'].unique())
overlap = train_clients_seen & test_clients_seen
print(f"\nClients appearing in BOTH train and test under the naive split: {len(overlap)} of {df['client_hash_id'].nunique()}")

BEFORE -- naive row-level random split (ignores client grouping)
[NAIVE random split] train n=145,708  test n=36,427
[NAIVE random split] train_sil=0.371  test_sil=0.375  gap=-0.004

AFTER -- honest split, grouped by client_hash_id (same as w05)
[GROUPED (honest) split] train n=161,153  test n=20,982
[GROUPED (honest) split] train_sil=0.373  test_sil=0.335  gap=+0.038

               split_type  train_sil  test_sil  train_minus_test_gap
    Naive random (before)   0.370987  0.374996             -0.004009
Grouped by client (after)   0.373040  0.335484              0.037556

Clients appearing in BOTH train and test under the naive split: 55 of 59


- Under the naive split, test silhouette (0.375) is slightly higher than train (0.371) — there is no generalization gap at all. That is not because the clusters generalize well; it is because 55 of 59 clients appear on both sides of the split. The model is effectively being tested on content from clients it already trained on.

- Under the grouped split, a real generalization gap appears: train 0.373 vs. test 0.335, a 0.038 drop when the model sees content from 12 entirely unseen clients. This is the honest number that shows how well the cluster structure holds up on clients the model has never touched.

- Takeaway: the naive split does not just produce a different number, it produces a falsely reassuring one. It hides the exact risk (client-specific memorization) that the grouped split exists to catch. Silhouette alone cannot tell you whether a split is honest, you have to check the split design itself before trusting the metric it produces.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.